<a href="https://colab.research.google.com/github/FinalInvictus/Deep-Mutual-Learning-with-Semi-Aves-Dataset/blob/Sridhar/DeepMutualLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Enter your Kaggle API token: ")

Enter your Kaggle API token: ··········


# Google Drive Mounting for Shared access

In [8]:
from google.colab import drive
drive.mount('/content/SharedData')

ValueError: Mountpoint must not already contain files

In [7]:
import kagglehub

path = kagglehub.competition_download(
    "semi-inat-2020",
    output_dir="/content/SharedData/MyDrive/Semi_Aves_Dataset",
    force_download=True
)

100%|██████████| 14.3G/14.3G [02:35<00:00, 98.8MB/s]

Extracting files...


# Parameter Initialization


In [ ]:
import torch
import torch.nn as nn

from torchvision.models import (
    resnet18,
    resnet34,
    ResNet18_Weights,
    ResNet34_Weights,
)

In [ ]:
class SemiAvesResNet(nn.Module):
    def __init__(
        self,
        architecture="resnet18",
        num_classes=200,
        freeze_backbone=False,
    ):
        super().__init__()
        self.architecture = architecture.lower()
        self.num_classes = num_classes
        self.freeze_backbone = freeze_backbone
        if self.architecture == "resnet18":
            self.backbone = resnet18(
                weights=ResNet18_Weights.IMAGENET1K_V1
            )
        elif self.architecture == "resnet34":
            self.backbone = resnet34(
                weights=ResNet34_Weights.IMAGENET1K_V1
            )
        else:
            raise ValueError(f"Not part of the architecture choices ")
        # 2. Read the feature size before removing the original head.
        # This is 512 for both ResNet-18 and ResNet-34.
        self.feature_dim = self.backbone.fc.in_features

        # 3. Remove the original 1,000-class ImageNet classifier.
        # The backbone now returns a 512-dimensional feature vector.
        self.backbone.fc = nn.Identity()

        # 4. Add a new classifier for the Semi-Aves classes.
        # This layer starts with randomly initialized parameters.
        self.classifier = nn.Linear(
            in_features=self.feature_dim,
            out_features=num_classes,
        )

        # 5. Optionally freeze the pretrained feature extractor.
        if self.freeze_backbone:
            for parameter in self.backbone.parameters():
                parameter.requires_grad = False

            self.backbone.eval()

    def forward_features(self, images):
        """Return the pretrained backbone's feature vectors."""
        return self.backbone(images)

    def forward(self, images):
        """Return raw class scores, called logits."""
        features = self.forward_features(images)
        logits = self.classifier(features)
        return logits

    def train(self, mode=True):
        """
        Switch model mode.

        If the backbone is frozen, keep its BatchNorm layers in
        evaluation mode so their running statistics stay fixed.
        """
        super().train(mode)

        if self.freeze_backbone:
            self.backbone.eval()

        return self